In [2]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

if GOOGLE_API_KEY:
    print("Gemini API key connected successfully!")
else:
    print("API key not found. Check Colab Secrets.")

Gemini API key connected successfully!


In [5]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded))

print("Uploaded file:", pdf_path)

Saving instructions.pdf to instructions.pdf
Uploaded file: instructions.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("PDF loaded successfully!")
print("Total pages:", len(pages))

print("\nFirst page preview:\n")
print(pages[0].page_content[:1000])

PDF loaded successfully!
Total pages: 1

First page preview:

Instructions for Student Discipline and Conduct  
During MSBTE Practical Examinations Winter-2026 
 
All students appearing for the MSBTE Practical Examination Winter-2026,  must strictly follow the instructions 
given below. 
1. Reporting Time: Students must report to the examination centre at least 30 minutes before the scheduled 
examination time.  The practical examination will be conducted batch -wise. Students whose practical 
examination is not scheduled in the current batch must remain seated in their allotted classroom and 
maintain silence and discipline. Students should proceed to the laboratory only when their batch is called 
by the concerned faculty or examiner. Students must not leave the classroom or enter the laboratory 
without permission. 
2. Identity Card: Students must carry their valid college identity card and MSBTE examination hall ticket. 
3. Attendance: Students must maintain regular attendance and 

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(pages)

print("Document split successfully!")
print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0].page_content)

Document split successfully!
Total chunks: 4

First chunk:

Instructions for Student Discipline and Conduct  
During MSBTE Practical Examinations Winter-2026 
 
All students appearing for the MSBTE Practical Examination Winter-2026,  must strictly follow the instructions 
given below. 
1. Reporting Time: Students must report to the examination centre at least 30 minutes before the scheduled 
examination time.  The practical examination will be conducted batch -wise. Students whose practical 
examination is not scheduled in the current batch must remain seated in their allotted classroom and 
maintain silence and discipline. Students should proceed to the laboratory only when their batch is called 
by the concerned faculty or examiner. Students must not leave the classroom or enter the laboratory 
without permission. 
2. Identity Card: Students must carry their valid college identity card and MSBTE examination hall ticket. 
3. Attendance: Students must maintain regular attendance and si

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

test_embedding = embeddings.embed_query(
    "What is this document about?"
)

print("Embeddings generated successfully!")
print("Embedding dimensions:", len(test_embedding))

Embeddings generated successfully!
Embedding dimensions: 3072


In [9]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="internship_rag",
    persist_directory="./chroma_db"
)

print("Document stored successfully in ChromaDB!")

Document stored successfully in ChromaDB!


In [10]:
question = "What is this document about?"

retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)

print("Retrieved relevant information:\n")

for i, doc in enumerate(retrieved_docs, 1):
    page_number = doc.metadata.get("page", "Unknown")

    print(f"\nResult {i}")
    print(f"Page: {page_number + 1 if isinstance(page_number, int) else page_number}")
    print(doc.page_content[:500])

Retrieved relevant information:


Result 1
Page: 1
Instructions for Student Discipline and Conduct  
During MSBTE Practical Examinations Winter-2026 
 
All students appearing for the MSBTE Practical Examination Winter-2026,  must strictly follow the instructions 
given below. 
1. Reporting Time: Students must report to the examination centre at least 30 minutes before the scheduled 
examination time.  The practical examination will be conducted batch -wise. Students whose practical 
examination is not scheduled in the current batch must remain s

Result 2
Page: 1
examination. 
11. Malpractice: Any form of copying, unfair practice, or violation of examination rules is strictly prohibited and 
will be dealt with as per the applicable examination regulations. 
12. Use of Equipment: Students must not operate any equipment without the examiner's or laboratory staff's 
permission. Any damage or malfunction must be reported immediately. 
13. Submission of Work: Students must submit their answ

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2
)

print("Gemini language model connected successfully!")

Gemini language model connected successfully!


In [12]:
def ask_question(question):

    # Retrieve relevant document chunks
    retrieved_docs = vector_store.similarity_search(
        question,
        k=3
    )

    if not retrieved_docs:
        return {
            "answer": "No relevant information found in the document.",
            "sources": []
        }

    # Prepare document context
    context = ""

    sources = []

    for doc in retrieved_docs:

        page = doc.metadata.get("page", "Unknown")

        if isinstance(page, int):
            page = page + 1

        context += f"""
        Source Page: {page}
        Content: {doc.page_content}
        """

        if page not in sources:
            sources.append(page)

    # Create prompt
    prompt = f"""
    You are a helpful document question-answering assistant.

    Answer the user's question using ONLY the provided context.

    Rules:
    1. Do not invent facts.
    2. If the answer is not available in the context,
       clearly say that the document does not contain the answer.
    3. Give a clear and simple answer.
    4. Mention relevant source page numbers.

    DOCUMENT CONTEXT:
    {context}

    USER QUESTION:
    {question}

    ANSWER:
    """

    # Generate answer
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "sources": sources
    }

print("RAG question-answering function created!")

RAG question-answering function created!


In [14]:
question = "What is this document about?"

result = ask_question(question)

print("QUESTION:")
print(question)

print("\nAI ANSWER:")
print(result["answer"])

print("\nSOURCE PAGES:")
print(result["sources"])

QUESTION:
What is this document about?

AI ANSWER:
[{'type': 'text', 'text': 'This document provides instructions for student discipline and conduct during the MSBTE Practical Examinations Winter-2026, outlining rules regarding reporting times, required materials, dress code, attendance, and general behavior (Source Page: 1).', 'extras': {'signature': 'EqMKCqAKAWkUfRPdVHWestvx+01h9IUSkWu797WOlvYucDUq/cVRymdjF1RXcg2W2HJVUeggvw/GQIHBHnn17D3Px9WDCQsb1QKpN/HZd6XdwR0RqMwecp8Yp+8up9ZmS1shJPp74ofAp5EEnXr2g/VIMrDd+QUSiMVKBKeuoFpnGxThL5N+55f9nDb9rWPOlCc+8pbJJ558xYLIzXmYBOZJwl41FOpKmXKWf4gxhhLuIAMvjdiQEMz3DDumnDOgYgizHxI4IjGXS4QUM53BY5ciWKeMueRSmWkqzK75/k1lMN1PqPZ6L7frsje2RuUvDqxJqFukxf8J5ilkxO7ZiUlGragdrJUnwNIczQnHRoyz7r6/2Uf3NSwFY7k4k550eH5aRNfkUwiXoqzBZVFsW8BJgWlhIC+uaxGgbeYGohOy7PIkYqFvGOW98QZCGQ+NAY7qq9aOHEy4Bs+tdQtS8JVtOm/6C7ciL5a/S2BhuM1N69+qD+m77j6cLOFWLeSV2Ld4OSBEi1Ar8pZcLtoAzys4doHGcYNcf2UNqbNRKXvf5JSsxbysis+osl7FwBKJ2flRH9M1EQJynE39xSRJfqaD84DQv6nK7MnlrvQt8oSzHY1agYbRcatX4Vq5wzLlMK63O

In [15]:
test_questions = [
    "What is this document about?",
    "What are the main points discussed?",
    "What important details are mentioned?",
    "What is the conclusion of this document?"
]

for question in test_questions:

    result = ask_question(question)

    print("\n" + "=" * 60)
    print("QUESTION:", question)
    print("\nANSWER:", result["answer"])
    print("\nSOURCE PAGES:", result["sources"])


QUESTION: What is this document about?

ANSWER: [{'type': 'text', 'text': 'Based on the document, it contains the instructions and guidelines for student discipline and conduct during the MSBTE Practical Examinations Winter-2026 (Source Page: 1).', 'extras': {'signature': 'EsEICr4IAWkUfRNAGmAgWS8DzAt9YYn81jhWIXiNN5eZjZ2FdVuvlG5k3u59BQw5htqjzJqqhRYlVks3slmQtaxIoAgV2/QOoah24vcytKEymG38cBw2+CqH+4KeUQYdtiga8TdDNEGeRqkIbfBOu1JdFAFgTRm4k/Aq7xqlz38K71nqEKBuN7AGdF3cj9DqpwIzDfrll5SCCdzydP3MN/5WC0C6Kz06vtvPqNfwt1JdvBjsbN79tLLU15cNxGqjOSxb7pRaC3BNBW0kyHD6aEMLmkEOwwlZIr1xq7D1krP4/Fa0sUL3ef3A9FC/H3JFKF+AlwBYTqBIa9HcgqInyONXYeVqJ9tNzIUUx09IJ6x8XSIauQtHSyWCtHmgF6na0Z1soRAbcNVdZcQsKi3dMc1es3r0NfVHw3yznQTBkis3LjRAeOPYWWIVCkoGWGrdn7s3jYsuUcQQh6FYWWNKcssDi8gZqXmz8zpGfP+2mfSwUCjgcqTYZvXFolFtgXRiAUOJG7+NFOkymmkSixmCjsTxLq+dYbcpXRsBkUt4DadKyX/MRpu2rut54r2kTDxZ2T+R5RMMRPah6fJ9ilnTXAGMmMItMsSaUHk1TlKQOetqgIVFG5cqNUcFuLBB+Ndflk9q4W6TAsYBGqk4qZQteKxCLIU+Ia1003j+i7L1hLGpV4Z9iRw3qWxLZbcUMTnuzoW6nNBEwAwmY0iGSiz+4

In [16]:
question = "What is the exact price of a spaceship mentioned in this document?"

result = ask_question(question)

print("QUESTION:", question)
print("ANSWER:", result["answer"])
print("SOURCES:", result["sources"])

QUESTION: What is the exact price of a spaceship mentioned in this document?
ANSWER: [{'type': 'text', 'text': 'The provided document does not contain any information about the price of a spaceship.', 'extras': {'signature': 'EoIICv8HAWkUfROH9VCrhe+tbfK2V8cYovlChMhza6/UcaTbkmw4G034rc8CWU6GPSzXM8+KtRm5fgUeLk9U7hUTVGMb6As25z6Jyr4rGDI1M5W+J0QUsgrnLIhiXfpTnbuKy5lspfHBkhgt3aRJQM0WQQx/AVs7X0YFZMyqcuqUGJeQQNnz49po3s1Clde20JQCT/KV36n34AG5VZwjzda8ALXsrderkMU8sGkXQnh24lUa51UdoW0uGkNTsyjnH9+z0Rg75EROWXm9cFSFlPp82JQ9uNVImuRVTUfP5IfFX/S/IiR5ZwtDL/NX0e7BPeesGK32SoUJb8kUftW93vePgQ9RHs6SjbZev0XUmsZ0anZFirhQBMfjbUkGwrk64JQUGniB9wOd1HjAjQzqLptsZ6ECPy7i+HCmolgJtKPYWlHrsytDw15XOcMJjk4oanVL4YaNWwRtsmirqFbaQU+CdwYVOhsF8m8rmVFvNtwOKS2S7ktbdfhC0rjlZeC4tUpj8V+voCn5g0Vy8JYxT7WS17bb4MMlyuDS7q0TP1KyMptR+vi/1rcT7z9yY2I1cF0eDUHjRh7HIs/cydcwjN7uWyBPM7Ffv8KOJ49XrAj2IOWsRbO7gop5H1tqG/BR//cVHeazEEB4z2UZ6sEpIUL4kAJmumEtzMee6kgYTtuVxVMpjem/f2kby2OwaMeQaD3RwOP2QZ5Pz4sUkYxrdy673jHCOkOdLOr2qeT7YbvkxDaxEJxi2WkpizwRzjRuhl/Rs0

In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Text(
    placeholder="Ask something about your PDF...",
    description="Question:",
    layout=widgets.Layout(width="80%")
)

ask_button = widgets.Button(
    description="Ask AI",
    button_style="success"
)

output = widgets.Output()

def handle_question(button):
    with output:
        clear_output()

        question = question_box.value.strip()

        if not question:
            print("Please enter a question.")
            return

        print("Thinking...\n")

        result = ask_question(question)

        print("AI ANSWER:")
        print(result["answer"])

        print("\nSOURCE PAGES:")
        print(result["sources"])

ask_button.on_click(handle_question)

display(question_box, ask_button, output)

Text(value='', description='Question:', layout=Layout(width='80%'), placeholder='Ask something about your PDF.…

Button(button_style='success', description='Ask AI', style=ButtonStyle())

Output()

In [19]:
chat_history = []

def chat_with_pdf(question):

    result = ask_question(question)

    chat_history.append({
        "question": question,
        "answer": result["answer"],
        "sources": result["sources"]
    })

    return result

print("Chat history system initialized!")

Chat history system initialized!


In [20]:
response = chat_with_pdf("What is this document about?")

print("Answer:", response["answer"])

print("\nConversation history:")
for chat in chat_history:
    print("\nQuestion:", chat["question"])
    print("Answer:", chat["answer"])

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 18h53m59.744849266s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 

In [21]:
evaluation_questions = [
    "What is this document about?",
    "What are the main topics?",
    "What important information is included?"
]

evaluation_results = []

for question in evaluation_questions:

    result = ask_question(question)

    evaluation_results.append({
        "question": question,
        "answer": result["answer"],
        "source_pages": result["sources"],
        "has_answer": bool(result["answer"])
    })

for item in evaluation_results:
    print("=" * 60)
    print("Question:", item["question"])
    print("Answer:", item["answer"])
    print("Source pages:", item["source_pages"])
    print("Response generated:", item["has_answer"])

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 18h52m1.34364864s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'G

In [22]:
import json

with open("rag_evaluation_results.json", "w") as file:
    json.dump(evaluation_results, file, indent=4)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!
